In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)], check=True)
    print("Cloned 'running': ", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)

for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src", "src", "../src"]:
    if Path(_p).exists():
        sys.path.insert(0, _p)
        print("Using src from:", _p)
        break

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Load target + star metadata

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

targets = pd.read_csv(DATA_ROOT / "train.csv")
wl_cols = [c for c in targets.columns if c != "planet_id"]
Y = targets[wl_cols].to_numpy(dtype=float)
print("targets:", Y.shape, "| planets:", targets.shape[0], "| wavelengths:", len(wl_cols))

star_path = DATA_ROOT / "train_star_info.csv"
star = pd.read_csv(star_path) if star_path.exists() else None
print("star_info:", None if star is None else star.shape)


## 2. Mean and std spectrum + a few sample spectra

In [ ]:
mean_spec = Y.mean(axis=0)
std_spec = Y.std(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(mean_spec, color="navy", label="mean")
ax[0].fill_between(np.arange(len(mean_spec)), mean_spec - std_spec, mean_spec + std_spec,
                   alpha=0.3, color="navy", label="+/-1 std")
ax[0].set_title("Mean spectrum +/- std")
ax[0].set_xlabel("wavelength index")
ax[0].set_ylabel("(Rp/Rs)^2")
ax[0].legend()
rng = np.random.default_rng(0)
for i in rng.choice(Y.shape[0], size=min(6, Y.shape[0]), replace=False):
    ax[1].plot(Y[i], lw=1)
ax[1].set_title("Sample target spectra")
ax[1].set_xlabel("wavelength index")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_target_spectra.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Distribution of transit depth

In [ ]:
depth_per_planet = Y.mean(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(depth_per_planet, bins=50, color="teal")
ax[0].set_title("Mean transit depth / planet")
ax[0].set_xlabel("(Rp/Rs)^2")
ax[1].hist(std_spec, bins=50, color="darkorange")
ax[1].set_title("Per-wavelength std (signal magnitude)")
ax[1].set_xlabel("std")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_depth_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"depth mean={depth_per_planet.mean():.4e} | spectral variation (median per-wl std)={np.median(std_spec):.4e}")


## 4. PCA explained variance

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=min(80, Y.shape[1], Y.shape[0])).fit(Y)
cum = np.cumsum(pca.explained_variance_ratio_)
for thr in (0.90, 0.95, 0.99):
    k = int(np.searchsorted(cum, thr) + 1)
    print(f"  cần {k:3d} components để giữ {thr*100:.0f}% variance")
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(cum) + 1), cum, "o-", ms=3)
for thr in (0.90, 0.95, 0.99):
    plt.axhline(thr, ls="--", lw=0.7, color="grey")
plt.xlabel("n_components")
plt.ylabel("cumulative explained variance")
plt.title("Target PCA - explained variance")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_pca_variance.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Raw light curve (physical signal)

In [ ]:
from config import PreprocessConfig, DatasetConfig
from data_io import ArielDataRepository
from pipeline import ArielPreprocessFeaturePipeline

repo = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
planets = repo.list_planet_ids("train")
if planets:
    pid = planets[0]
    pipe = ArielPreprocessFeaturePipeline(PreprocessConfig(target_time_bins=128, apply_cds=True, detrend_degree=2, smooth_window=5))
    obs = repo.load_observation("train", pid)
    curves = pipe.processed_light_curves(obs, airs_adc=repo.get_adc_params("AIRS-CH0", pid), fgs_adc=repo.get_adc_params("FGS1", pid))
    airs = np.nan_to_num(curves.airs)
    fgs = np.nan_to_num(curves.fgs)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(fgs, color="black")
    ax[0].set_title(f"FGS1 white-light (planet {pid})")
    ax[0].set_xlabel("time bin")
    for j in np.linspace(0, airs.shape[1]-1, 5, dtype=int):
        ax[1].plot(airs[:, j], lw=1, label=f"wl {j}")
    ax[1].set_title("AIRS light curves")
    ax[1].set_xlabel("time bin")
    ax[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_light_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No raw train data found — skipping light curve.")


## 6. Star metadata + missing/outlier

In [ ]:
if star is not None:
    num = star.select_dtypes("number").drop(columns=[c for c in ["planet_id"] if c in star.columns], errors="ignore")
    print(num.describe().T[["mean", "std", "min", "max"]])
    num.hist(bins=30, figsize=(11, 6))
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_star_metadata.png", dpi=150, bbox_inches="tight")
    plt.show()

print("\nNaN trong target:", int(np.isnan(Y).sum()))
z = np.abs((depth_per_planet - depth_per_planet.mean()) / (depth_per_planet.std() + 1e-12))
print("Outlier planet (|z|>4) theo mean depth:", int((z > 4).sum()))
